In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# =========================================================
# LOAD CSV DATA
# =========================================================

baca = pd.read_csv("baca_results.csv")
aca = pd.read_csv("aca_results.csv")

# =========================================================
# CREATE FIGURE
# =========================================================

fig = go.Figure()

# =========================================================
# BACA ANALYSIS
# =========================================================

d_values = sorted(baca["d"].unique())

for d in d_values:

    # -----------------------------------------------------
    # FILTER d
    # -----------------------------------------------------

    df = baca[baca["d"] == d].copy()

    # -----------------------------------------------------
    # GROUP BY n
    #
    # We now compute robust statistics because
    # every repeat is stored individually.
    # -----------------------------------------------------

    grouped = (
        df.groupby("n")
        .agg({
            "elapsed": ["median", "mean", "std"],
            "rank": ["median", "mean", "std"],
            "T_over_rank": ["median", "mean", "std"],
            "T_over_nk": ["median", "mean", "std"],
            "T_over_nk2": ["median", "mean", "std"]
        })
    )

    # flatten multi-index columns
    grouped.columns = [
        "_".join(col).strip()
        for col in grouped.columns.values
    ]

    grouped = grouped.reset_index()

    # -----------------------------------------------------
    # USE MEDIAN T/rank
    #
    # Median is MUCH more stable for randomized
    # algorithms than mean.
    # -----------------------------------------------------

    x = grouped["n"].to_numpy()

    y = grouped["T_over_rank_median"].to_numpy()

    y_std = grouped["T_over_rank_std"].to_numpy()

    # -----------------------------------------------------
    # RAW MEASUREMENTS (faint)
    # -----------------------------------------------------

    fig.add_trace(
        go.Scatter(
            x=df["n"],
            y=df["T_over_rank"],
            mode='markers',
            opacity=0.25,
            marker=dict(size=5),
            name=f'BACA d={d} raw samples'
        )
    )

    # -----------------------------------------------------
    # MEDIAN + ERROR BARS
    # -----------------------------------------------------

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode='markers+lines',
            error_y=dict(
                type='data',
                array=y_std,
                visible=True
            ),
            marker=dict(size=10),
            name=f'BACA d={d} median'
        )
    )

    # -----------------------------------------------------
    # LINEAR FIT TO MEDIAN
    # -----------------------------------------------------

    coeff_lin = np.polyfit(x, y, 1)

    fit_lin = np.poly1d(coeff_lin)

    x_fit = np.linspace(
        np.min(x),
        np.max(x),
        300
    )

    fig.add_trace(
        go.Scatter(
            x=x_fit,
            y=fit_lin(x_fit),
            mode='lines',
            name=f'BACA d={d} linear fit'
        )
    )

    # -----------------------------------------------------
    # QUADRATIC FIT TO MEDIAN
    # -----------------------------------------------------

    coeff_quad = np.polyfit(x, y, 2)

    fit_quad = np.poly1d(coeff_quad)

    fig.add_trace(
        go.Scatter(
            x=x_fit,
            y=fit_quad(x_fit),
            mode='lines',
            visible='legendonly',
            name=f'BACA d={d} quadratic fit'
        )
    )

    # -----------------------------------------------------
    # PRINT STATS
    # -----------------------------------------------------

    print("\n================================================")
    print(f"BACA d = {d}")
    print("================================================")

    print("\nMedian T/rank values:")
    print(grouped[["n", "T_over_rank_median"]])

    print("\nStd(T/rank):")
    print(grouped[["n", "T_over_rank_std"]])

    print("\nLinear coefficients:")
    print(coeff_lin)

    print("\nQuadratic coefficients:")
    print(coeff_quad)

# =========================================================
# ACA ANALYSIS
# =========================================================

grouped_aca = (
    aca.groupby("n")
    .agg({
        "elapsed": ["median", "mean", "std"],
        "rank": ["median", "mean", "std"],
        "T_over_rank": ["median", "mean", "std"],
        "T_over_nk": ["median", "mean", "std"],
        "T_over_nk2": ["median", "mean", "std"]
    })
)

grouped_aca.columns = [
    "_".join(col).strip()
    for col in grouped_aca.columns.values
]

grouped_aca = grouped_aca.reset_index()

# ---------------------------------------------------------
# MEDIAN DATA
# ---------------------------------------------------------

x_aca = grouped_aca["n"].to_numpy()

y_aca = grouped_aca["T_over_rank_median"].to_numpy()

y_std_aca = grouped_aca["T_over_rank_std"].to_numpy()

# ---------------------------------------------------------
# RAW ACA SAMPLES
# ---------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=aca["n"],
        y=aca["T_over_rank"],
        mode='markers',
        opacity=0.25,
        marker=dict(size=5),
        name='ACA raw samples'
    )
)

# ---------------------------------------------------------
# ACA MEDIAN + ERROR BARS
# ---------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=x_aca,
        y=y_aca,
        mode='markers+lines',
        error_y=dict(
            type='data',
            array=y_std_aca,
            visible=True
        ),
        marker=dict(size=10),
        name='ACA median'
    )
)

# ---------------------------------------------------------
# ACA LINEAR FIT
# ---------------------------------------------------------

coeff_lin_aca = np.polyfit(x_aca, y_aca, 1)

fit_lin_aca = np.poly1d(coeff_lin_aca)

x_fit_aca = np.linspace(
    np.min(x_aca),
    np.max(x_aca),
    300
)

fig.add_trace(
    go.Scatter(
        x=x_fit_aca,
        y=fit_lin_aca(x_fit_aca),
        mode='lines',
        name='ACA linear fit'
    )
)

# ---------------------------------------------------------
# ACA QUADRATIC FIT
# ---------------------------------------------------------

coeff_quad_aca = np.polyfit(x_aca, y_aca, 2)

fit_quad_aca = np.poly1d(coeff_quad_aca)

fig.add_trace(
    go.Scatter(
        x=x_fit_aca,
        y=fit_quad_aca(x_fit_aca),
        mode='lines',
        visible='legendonly',
        name='ACA quadratic fit'
    )
)

# =========================================================
# PRINT ACA STATS
# =========================================================

print("\n================================================")
print("ACA")
print("================================================")

print("\nMedian T/rank values:")
print(grouped_aca[["n", "T_over_rank_median"]])

print("\nStd(T/rank):")
print(grouped_aca[["n", "T_over_rank_std"]])

print("\nLinear coefficients:")
print(coeff_lin_aca)

print("\nQuadratic coefficients:")
print(coeff_quad_aca)

# =========================================================
# LAYOUT
# =========================================================

fig.update_layout(
    title="ACA vs BACA Runtime Scaling",
    xaxis_title="Matrix size n",
    yaxis_title="Median(T / rank)",
    template="plotly_white",
    hovermode="closest",
    width=1400,
    height=800
)

# =========================================================
# SHOW
# =========================================================

fig.show()



BACA d = 2

Median T/rank values:
       n  T_over_rank_median
0   1100            0.003155
1   1200            0.003729
2   1300            0.004789
3   1400            0.005448
4   1500            0.006874
5   1600            0.006824
6   1700            0.006439
7   1800            0.007934
8   1900            0.009623
9   2000            0.010643
10  2100            0.011896
11  2200            0.011696
12  2300            0.013792
13  2400            0.013597
14  2500            0.020433
15  2600            0.019857
16  2700            0.021484
17  2800            0.018668
18  2900            0.021682
19  3000            0.024506

Std(T/rank):
       n  T_over_rank_std
0   1100         0.000264
1   1200         0.000715
2   1300         0.001489
3   1400         0.001528
4   1500         0.001624
5   1600         0.001915
6   1700         0.001552
7   1800         0.001288
8   1900         0.002238
9   2000         0.002820
10  2100         0.002164
11  2200         0.001712
12  

In [2]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(
    "frobenius_norms.csv",
    names=["method", "d", "start", "rank", "frob"],
    header=0
)

df_svd = pd.read_csv(
    "frobenius_norms_svd.csv",
    names=["rank", "frob"],
    header=0
)

# =========================================================
# FORCE NUMERIC TYPES
# =========================================================

df["d"] = pd.to_numeric(df["d"])
df["start"] = pd.to_numeric(df["start"])
df["rank"] = pd.to_numeric(df["rank"])
df["frob"] = pd.to_numeric(df["frob"])

df_svd["rank"] = pd.to_numeric(df_svd["rank"])
df_svd["frob"] = pd.to_numeric(df_svd["frob"])

# =========================================================
# SORT
# =========================================================

df = df.sort_values(["method", "d", "rank"])

# =========================================================
# CREATE FIGURE
# =========================================================

fig = go.Figure()

# =========================================================
# ACA
# =========================================================

aca = df[df["method"] == "ACA"]

x = aca["rank"].to_numpy(dtype=float)
y = aca["frob"].to_numpy(dtype=float)

fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="lines+markers",
        name="ACA"
    )
)

# =========================================================
# BACA
# =========================================================

d_values = sorted(
    df[df["method"] == "BACA"]["d"].unique()
)

for d in d_values:

    sub = df[
        (df["method"] == "BACA") &
        (df["d"] == d)
    ]

    x = sub["rank"].to_numpy(dtype=float)
    y = sub["frob"].to_numpy(dtype=float)

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="lines+markers",
            name=f"BACA d = {d}"
        )
    )

# =========================================================
# SVD
# =========================================================

x = df_svd["rank"].to_numpy(dtype=float)
y = df_svd["frob"].to_numpy(dtype=float)

fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="lines+markers",
        name="SVD"
    )
)

# =========================================================
# LAYOUT
# =========================================================

fig.update_layout(
    title="Frobenius Error vs Rank of Gaussian Matrix",
    xaxis_title="Rank",
    yaxis_title="Frobenius Norm of Error",
    template="plotly_white",
    hovermode="x unified",
    width=1200,
    height=700
)

# =========================================================
# LOG SCALE
# =========================================================

fig.update_yaxes(type="log")

# =========================================================
# SHOW
# =========================================================

fig.show()

In [4]:
import json
import pandas as pd
from pathlib import Path

results_dir = Path("results")

files = list(results_dir.glob("*.json"))
print(len(files))

3


In [5]:
experiments = []

for file in files:
    with open(file) as f:
        data = json.load(f)

    experiments.append({
        "file": file.name,
        "timestamp": data.get("timestamp"),
        "method": data.get("method"),
        "test_function": data.get("test_function")
    })

exp_df = pd.DataFrame(experiments)
exp_df

,file,timestamp,method,test_function
0,2026-05-29_17-31-26.json,results/2026-05-29_17-31-26.json,ACA,test_function_gaussian
1,2026-05-29_17-31-41.json,results/2026-05-29_17-31-41.json,BACA,test_function_gaussian
2,2026-05-29_17-32-33.json,results/2026-05-29_17-32-33.json,HBACA,test_function_gaussian


In [6]:
runtime_rows = []

for file in files:
    with open(file) as f:
        data = json.load(f)

    if "runtime_benchmark" not in data:
        continue

    method = data["method"]

    for case in data["runtime_benchmark"]["cases"]:

        agg = case["aggregated_runs"]

        runtime_rows.append({
            "method": method,
            "n": case["n"],
            "rows": case["rows"],
            "cols": case["cols"],
            "avg_time": agg["avg_time"],
            "avg_rank": agg["avg_rank"]
        })

runtime_df = pd.DataFrame(runtime_rows)
runtime_df

,method,n,rows,cols,avg_time,avg_rank
0,ACA,1100,1100,1100,0.00850,1.00
1,ACA,1200,1200,1200,0.01000,1.00
2,ACA,1300,1300,1300,0.01200,1.00
3,ACA,1400,1400,1400,0.01425,1.00
4,ACA,1500,1500,1500,0.01575,1.00
...,...,...,...,...,...,...
115,HBACA,4600,4600,4600,1.15850,7.00
116,HBACA,4700,4700,4700,0.60500,7.00
117,HBACA,4800,4800,4800,0.56700,7.25
118,HBACA,4900,4900,4900,0.64475,7.00


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display, clear_output
from plotly.subplots import make_subplots

# =====================================================
# FIND FILES
# =====================================================

results_dir = Path("results")
files = sorted(results_dir.glob("*.json"))

if not files:
    raise ValueError("No JSON files found")

# =====================================================
# BENCHMARK TYPE
# =====================================================

def get_benchmark_type(data):

    if "runtime_benchmark" in data:
        return "runtime"

    if "residual_benchmark" in data:
        return "residual"

    return "unknown"


# =====================================================
# BUILD FILE LABELS
# =====================================================

file_meta = {}

for f in files:

    try:

        with open(f) as fp:
            data = json.load(fp)

        method = data.get("method", "?")

        if "runtime_benchmark" in data:

            runtime = data["runtime_benchmark"]

            eps = runtime.get("epsilon", "?")
            repeats = runtime.get("repeats", "?")

            d = data.get("d", "?")
            L = data.get("L", "?")

            nvals = [
                c.get("n")
                for c in runtime.get("cases", [])
            ]

            if nvals:
                nstr = f"{min(nvals)}–{max(nvals)}"
            else:
                nstr = "?"

            label = (
                f"[RUNTIME] "
                f"{method:6s} "
                f"d={d} "
                f"L={L} "
                f"eps={eps} "
                f"rep={repeats} "
                f"n=[{nstr}] "
                f"{f.name}"
            )

        elif "residual_benchmark" in data:

            residual = data["residual_benchmark"]

            eps = residual.get("epsilon", "?")

            label = (
                f"[RESIDUAL] "
                f"{method:6s} "
                f"eps={eps} "
                f"{f.name}"
            )

        else:

            label = f"[UNKNOWN] {f.name}"

    except Exception:

        label = f"[BROKEN] {f.name}"

    file_meta[f] = label


# =====================================================
# CHECKBOXES
# =====================================================

checkboxes = []

for f in files:

    cb = widgets.Checkbox(
        value=False,
        description=file_meta[f],
        indent=False,
        layout=widgets.Layout(width="95%")
    )

    checkboxes.append(cb)

show_std = widgets.Checkbox(
    value=True,
    description="Show std"
)

show_medians = widgets.Checkbox(
    value=True,
    description="Show median lines"
)

plot_button = widgets.Button(
    description="Plot",
    button_style="success"
)

select_all = widgets.Button(
    description="Select All"
)

clear_all = widgets.Button(
    description="Clear"
)

output = widgets.Output()

display(
    widgets.VBox([
        widgets.HTML("<h3>Select benchmark files</h3>"),
        widgets.HBox([
            select_all,
            clear_all,
            plot_button,
            show_medians,
            show_std
        ]),
        widgets.VBox(checkboxes),
        output
    ])
)

# =====================================================
# BUTTON HELPERS
# =====================================================

def do_select_all(_):

    for cb in checkboxes:
        cb.value = True


def do_clear(_):

    for cb in checkboxes:
        cb.value = False


select_all.on_click(do_select_all)
clear_all.on_click(do_clear)

# =====================================================
# RUNTIME LOADER
# =====================================================

def load_runtime_json(filename):

    with open(filename) as fp:
        data = json.load(fp)

    runtime = data.get("runtime_benchmark")

    rows = []

    for case in runtime["cases"]:

        n = case["n"]

        for run in case["runs"]:

            rows.append({
                "benchmark_type": "runtime",
                "file": Path(filename).stem,
                "method": data["method"],
                "d": data.get("d", np.nan),
                "L": data.get("L", np.nan),
                "n": n,
                "elapsed": run["elapsed"],
                "rank": run["rank"],
                "T_over_rank": run["T_over_rank"]
            })

    return pd.DataFrame(rows)


# =====================================================
# RESIDUAL TREE WALK
# =====================================================

def extract_residual_rows(
    node,
    rows,
    meta,
    rank_counter
):

    if node is None:
        return

    children = node.get("children", [])

    is_leaf = (
        len(children) == 0
        or all(c is None for c in children)
    )

    if is_leaf:

        rank_inc = node.get("rank_inc", [])
        uvals = node.get("u", [])
        vvals = node.get("v", [])

        for r, u, v in zip(
            rank_inc,
            uvals,
            vvals
        ):

            rank_counter[0] += r

            rows.append({
                **meta,
                "benchmark_type": "residual",
                "rank": rank_counter[0],
                "u": u,
                "v": v
            })

        return

    for child in children:

        extract_residual_rows(
            child,
            rows,
            meta,
            rank_counter
        )


# =====================================================
# RESIDUAL LOADER
# =====================================================

def load_residual_json(filename):

    with open(filename) as fp:
        data = json.load(fp)

    residual = data.get("residual_benchmark")
    d = data.get("d", np.nan)
    L = data.get("L", np.nan)
    if residual is None:
        return pd.DataFrame()

    rows = []

    starts = residual.get("starts", {})

    if isinstance(starts, dict):
        iterator = starts.items()
    else:
        iterator = enumerate(starts)

    for start_key, start_data in iterator:

        residuals_v = start_data.get(
            "residuals_v",
            []
        )

        residuals_u = start_data.get(
            "residuals_u",
            []
        )

        frob_norms = start_data.get(
            "frob_norms",
            []
        )

        method = data.get("method", "")

        # =====================================
        # BACA
        # =====================================

        if "rank_increase" in start_data:

            rank_increase = start_data["rank_increase"]

            cumulative_rank = 0

            # -----------------------------
            # v residuals
            # -----------------------------

            n_v = min(
                len(rank_increase),
                len(residuals_v)
            )


            for i in range(n_v):

                r = rank_increase[i]

                if r <= 0:
                    break

                cumulative_rank += r

                rows.append({
                    "method": method,
                    "d": d,
                    "L": L,
                    "start": start_key,
                    "benchmark_type": "residual",
                    "file": Path(filename).stem,
                    "method": method,
                    "start": start_key,
                    "series": "v",
                    "rank": cumulative_rank,
                    "value": residuals_v[i] / residuals_u[i],
        })

            # -----------------------------
            # Frobenius norm
            # -----------------------------

            cumulative_rank = 0

            for r in rank_increase:

                if r <= 0:
                    break

                cumulative_rank += r

                if cumulative_rank <= len(frob_norms):

                    rows.append({
                        "method": method,
                        "d": d,
                        "L": L,
                        "start": start_key,
                        "benchmark_type": "residual",
                        "file": Path(filename).stem,
                        "method": method,
                        "start": start_key,
                        "series": "frob",
                        "rank": cumulative_rank,
                        "value": frob_norms[int(cumulative_rank) - 1],
                    })  

        # =====================================
        # ACA
        # =====================================

        else:

            n = min(
                len(residuals_v),
                len(residuals_u),
                len(frob_norms)
            )

            for rank in range(1, n + 1):

                i = rank - 1

                rows.append({
                    "method": method,
                    "d": d,
                    "L": L,
                    "start": start_key,
                    "benchmark_type": "residual",
                    "file": Path(filename).stem,
                    "method": method,
                    "start": start_key,
                    "series": "v",
                    "rank": rank,
                    "value": (
                        residuals_v[i] / residuals_u[i]
                        if residuals_u[i] != 0
                        else np.nan
                    ),
                })

                rows.append({
                    "method": method,
                    "d": d,
                    "L": L,
                    "benchmark_type": "residual",
                    "file": Path(filename).stem,
                    "method": method,
                    "start": start_key,
                    "series": "frob",
                    "rank": rank,
                    "value": frob_norms[i],
                })

    return pd.DataFrame(rows)

# =====================================================
# PLOT
# =====================================================

def plot_clicked(_):

    with output:

        clear_output()

        selected = [
            files[i]
            for i, cb in enumerate(checkboxes)
            if cb.value
        ]

        if not selected:
            print("Select files.")
            return

        runtime_dfs = []
        residual_dfs = []

        for f in selected:

            try:

                with open(f) as fp:
                    data = json.load(fp)

                if "runtime_benchmark" in data:

                    df = load_runtime_json(f)

                    if not df.empty:
                        runtime_dfs.append(df)

                elif "residual_benchmark" in data:

                    df = load_residual_json(f)

                    if not df.empty:
                        residual_dfs.append(df)

            except Exception as e:

                print(f"{f.name}: {e}")
        # =============================================
        # RUNTIME PLOTS
        # =============================================

        if runtime_dfs:

            df = pd.concat(runtime_dfs)

            def method_family(method):

                m = str(method).lower()

                if m.startswith("hbaca"):
                    return "HBACA"

                if m.startswith("baca"):
                    return "BACA"

                if m.startswith("aca"):
                    return "ACA"

                return None

            # --------------------------------------------------
            # global scales
            # --------------------------------------------------

            stats_all = (
                df.groupby(
                    ["method", "d", "L", "file", "n"],
                    dropna=False
                )
                .agg({
                    "T_over_rank": "median",
                    "elapsed": "median"
                })
                .reset_index()
            )

            xmin = stats_all["n"].min()
            xmax = stats_all["n"].max()

            rank_vals = stats_all["T_over_rank"]
            rank_vals = rank_vals[rank_vals > 0]

            time_vals = stats_all["elapsed"]
            time_vals = time_vals[time_vals > 0]

            rank_range = [
                np.log10(rank_vals.min()),
                np.log10(rank_vals.max())
            ]

            time_range = [
                np.log10(time_vals.min()),
                np.log10(time_vals.max())
            ]

            # --------------------------------------------------
            # subplot figures
            # --------------------------------------------------

            fig_rank = make_subplots(
                rows=1,
                cols=3,
                subplot_titles=(
                    "ACA",
                    "BACA",
                    "HBACA"
                )
            )

            fig_time = make_subplots(
                rows=1,
                cols=3,
                subplot_titles=(
                    "ACA",
                    "BACA",
                    "HBACA"
                )
            )

            family_col = {
                "ACA": 1,
                "BACA": 2,
                "HBACA": 3,
            }

            for key, subdf in df.groupby(
                ["method", "d", "L", "file"],
                dropna=False
            ):

                method, d, L, fname = key

                family = method_family(method)

                if family is None:
                    continue

                col = family_col[family]

                stats = (
                    subdf.groupby("n")
                    .agg({
                        "T_over_rank": ["median", "std"],
                        "elapsed": ["median", "std"]
                    })
                )

                stats.columns = [
                    "_".join(c)
                    for c in stats.columns
                ]

                stats = stats.reset_index()

                label = f"{method.upper()} d={d} L={L}"

                fig_rank.add_trace(
                    go.Scatter(
                        x=stats["n"],
                        y=stats["T_over_rank_median"],
                        mode="lines+markers"
                        if show_medians.value
                        else "markers",
                        error_y=dict(
                            type="data",
                            array=stats[
                                "T_over_rank_std"
                            ].fillna(0),
                            visible=show_std.value
                        ),
                        name=label
                    ),
                    row=1,
                    col=col
                )

                fig_time.add_trace(
                    go.Scatter(
                        x=stats["n"],
                        y=stats["elapsed_median"],
                        mode="lines+markers"
                        if show_medians.value
                        else "markers",
                        error_y=dict(
                            type="data",
                            array=stats[
                                "elapsed_std"
                            ].fillna(0),
                            visible=show_std.value
                        ),
                        name=label
                    ),
                    row=1,
                    col=col
                )

            # --------------------------------------------------
            # identical axes
            # --------------------------------------------------

            fig_rank.update_xaxes(
                range=[xmin, xmax]
            )

            fig_rank.update_yaxes(
                type="log",
                range=rank_range
            )

            fig_time.update_xaxes(
                range=[xmin, xmax]
            )

            fig_time.update_yaxes(
                type="log",
                range=time_range
            )

            fig_rank.update_layout(
                title="Runtime Benchmark: T/rank",
                template="plotly_white",
                width=1800,
                height=600
            )

            fig_time.update_layout(
                title="Runtime Benchmark: elapsed",
                template="plotly_white",
                width=1800,
                height=600
            )

            fig_rank.show()
            fig_time.show()
        # =============================================
        # RESIDUAL PLOTS
        # =============================================

        if residual_dfs:

            df = pd.concat(residual_dfs)

            fig_v = go.Figure()
            fig_frob = go.Figure()

            for key, subdf in df.groupby(
                ["method", "d", "L", "file", "start"]
            ):

                method, d, L, fname, start = key

                vdf = (
                    subdf[subdf["series"] == "v"]
                    .sort_values("rank")
                )

                fdf = (
                    subdf[subdf["series"] == "frob"]
                    .sort_values("rank")
                )
                label = f"{method.upper()} d={d} L={L}"
                # V residual
                fig_v.add_trace(
                    go.Scatter(
                        x=vdf["rank"],
                        y=vdf["value"],
                        mode="lines+markers",
                        name=label
                    )
                )

                # Frobenius norm
                fig_frob.add_trace(
                    go.Scatter(
                        x=fdf["rank"],
                        y=fdf["value"],
                        mode="lines+markers",
                        name=label
                    )
                )

            fig_v.update_layout(
                title="Residual Benchmark: V Residual",
                xaxis_title="rank",
                yaxis_title="relative residual",
                template="plotly_white",
                width=1200,
                height=700
            )

            fig_v.update_yaxes(type="log")

            fig_frob.update_layout(
                title="Residual Benchmark: Frobenius Norm",
                xaxis_title="rank",
                yaxis_title="Frobenius norm",
                template="plotly_white",
                width=1200,
                height=700
            )

            fig_frob.update_yaxes(type="log")

            fig_v.show()
            fig_frob.show()


plot_button.on_click(plot_clicked)